#Load Data


In [ ]:
import numpy as np
import os

# Configuration
folder_path = 'Batch Processing Modal Parameter'
NUM_MODES_TO_EXTRACT = 3 
FEATURES_PER_MODE = 11

# Your specific filenames exactly as they appear in the folder
file_list = [
    "Modalparameters_UD_Group_final_all.npy",
    "Modalparameters_LD_E1_Group_final_all.npy",
    "Modalparameters_LD_E2_Group_final_all.npy",
    "Modalparameters_LD_E3_Group_final_all.npy",
    "Modalparameters_LD_E4_Group_final_all.npy",
    "Modalparameters_MD_E1_Group_final_all.npy",
    "Modalparameters_MD_E2_Group_final_all.npy",
    "Modalparameters_MD_E3_Group_final_all.npy",
    "Modalparameters_MD_E4_Group_final_all.npy",
    "Modalparameters_HD_E1_Group_final_all.npy",
    "Modalparameters_HD_E2_Group_final_all.npy",
    "Modalparameters_HD_E3_Group_final_all.npy",
    "Modalparameters_HD_E4_Group_final_all.npy"
]

In [4]:
def process_modal_file(full_path):
    data = np.load(full_path, allow_pickle=True)
    
    # 1. Ensure we have 3 modes
    current_modes = data.shape[1]
    if current_modes >= NUM_MODES_TO_EXTRACT:
        data_sliced = data[:, :NUM_MODES_TO_EXTRACT, :]
    else:
        padding = np.zeros((data.shape[0], NUM_MODES_TO_EXTRACT - current_modes, 11))
        data_sliced = np.concatenate([data, padding], axis=1)
    
    # 2. EXCLUDE DAMPING: 
    # Take index 0 (Freq) and indices 2-10 (Mode Shapes)
    # This removes index 1 (Damping)
    # New shape per mode: 1 (Freq) + 9 (MS) = 10 features
    indices_to_keep = [0] + list(range(2, 11))
    data_no_damp = data_sliced[:, :, indices_to_keep]
    
    # 3. Convert to Magnitude
    data_mag = np.abs(data_no_damp)
    
    # 4. Flatten: (Trials, 3 * 10) -> (Trials, 30)
    data_flattened = data_mag.reshape(data.shape[0], -1)
    
    return data_flattened

In [5]:
all_X = []
all_y = []

for idx, file_name in enumerate(file_list):
    full_path = os.path.join(folder_path, file_name)
    
    if os.path.exists(full_path):
        # Process and append
        X_class = process_modal_file(full_path)
        all_X.append(X_class)
        
        # Labeling (0 to 12)
        y_class = np.full(X_class.shape[0], idx)
        all_y.append(y_class)
        
        print(f"✅ Success: {file_name} | Rows: {X_class.shape[0]}")
    else:
        print(f"❌ Missing: {file_name} - Check folder path!")

# Combine into final matrices
X_final = np.vstack(all_X)
y_final = np.concatenate(all_y)

print("-" * 30)
print(f"Final Matrix Shape: {X_final.shape}") # Should be approx (780, 33)
print(f"Final Label Shape:  {y_final.shape}")

✅ Success: Modalparameters_UD_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_LD_E1_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_LD_E2_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_LD_E3_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_LD_E4_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_MD_E1_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_MD_E2_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_MD_E3_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_MD_E4_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_HD_E1_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_HD_E2_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_HD_E3_Group_final_all.npy | Rows: 60
✅ Success: Modalparameters_HD_E4_Group_final_all.npy | Rows: 60
------------------------------
Final Matrix Shape: (780, 30)
Final Label Shape:  (780,)


In [6]:
# Define your output filenames
x_filename = f"Concatenated Modal Parameter\\X_modal_features.npy"
y_filename = f"Concatenated Modal Parameter\\y_modal_labels.npy"

# Save the arrays
np.save(x_filename, X_final)
np.save(y_filename, y_final)

print(f"Successfully saved features to: {x_filename} ({X_final.shape})")
print(f"Successfully saved labels to:   {y_filename} ({y_final.shape})")

# Quick Verification: How to load them in your next notebook
# X_load = np.load('X_modal_features.npy')
# y_load = np.load('y_modal_labels.npy')

Successfully saved features to: Concatenated Modal Parameter\X_modal_features.npy ((780, 30))
Successfully saved labels to:   Concatenated Modal Parameter\y_modal_labels.npy ((780,))


Indices,Description for final data
0 - 10,"Mode 1: [Freq, Damping, MS1, MS2, ... MS9]"
11 - 21,"Mode 2: [Freq, Damping, MS1, MS2, ... MS9]"
22 - 32,"Mode 3: [Freq, Damping, MS1, MS2, ... MS9]"